<a href="https://colab.research.google.com/github/Abhi712-ui/lis501-ipynb-files/blob/main/Week_6_Topic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LIS 501 · Week 6
## Explore a news collection, then discover topic mixtures

**Question:** What recurring patterns of word use can we find in this collection, and how do they mix within articles?

We will work with **`Ukraine_News_Dataset.xlsx`**, which contains an `Articles` worksheet. Each row is one collected text. We first turn its citation into separate metadata columns. Then we inspect the collection, make a Bag-of-Words table, and fit a **Latent Dirichlet Allocation (LDA)** topic model.

**By the end:** you should be able to read a metadata table, describe an important limitation of the collection, inspect a topic's top words, and interpret one article's topic mixture.

| Part |
|---|
| Load the workbook and separate the metadata |
| Explore publishers, dates, and lengths; Exercise 1 |
| Check text quality and create word counts |
| Fit LDA and inspect its two output tables |
| Read topic mixtures and supporting articles; Exercise 2 |
| Save results and discuss limitations |
| **Main demonstration** |

Upload the workbook before starting the timed demonstration. Run cells from top to bottom. No subscription login, web scraping, API key, or language-model download is used.

**Content note:** this collection includes accounts of war, displacement, and violence. The first reading example concerns education. Preview another article before displaying it to the class. Titles and passages below are supplied source material, not independently verified factual statements.

### Setup
Open this notebook in Google Colab and upload `Ukraine_News_Dataset.xlsx` using the Files sidebar (folder icon). Keep that exact filename. A standard CPU runtime is sufficient.

The notebook uses pandas to work with tables, Matplotlib to plot, and scikit-learn to build word counts and LDA. Pandas 2.0 or later is needed for the mixed-format date conversion. If a local environment lacks the packages, install them before class with `%pip install pandas openpyxl scikit-learn matplotlib` in a separate setup cell.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from matplotlib.ticker import MaxNLocator
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation

pd.set_option("display.max_colwidth", 85)
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)

pandas: 2.2.3 | scikit-learn: 1.6.1


In [ ]:
file_name = "Ukraine_News_Dataset.xlsx"
if not Path(file_name).exists():
    raise FileNotFoundError("Upload Ukraine_News_Dataset.xlsx using Colab's Files sidebar.")

workbook = pd.ExcelFile(file_name)
print("Worksheets:", workbook.sheet_names)
raw = pd.read_excel(workbook, sheet_name="Articles")
print("Rows and columns:", raw.shape)
print("Column names:", raw.columns.tolist())
display(raw.head(3))

Worksheets: ['Articles']
Rows and columns: (67, 2)
Column names: ['Date', 'Article']


,Date,Article
0,"Rainsford, Sarah (6 September 2023). BBC News. ""Ukraine war: Romania reveals Russ...",Ukraine war: Romania reveals Russian drone parts hit its territory. Romania's def...
1,"MacFarquhar, Neil (17 March 2024). The New York Times. ""Five Takeaways From Putin...",President Vladimir V. Putin is expected to use the scale of his victory to justif...
2,"Goodman, Jack (2022-04-09). BBC News. ""Chernihiv: Are these Russia's weapons of w...","Chernihiv: Are these Russia's weapons of war? By Jack Goodman, Kayleen Devlin, Ma..."


## 1. One article per row; one kind of information per column

A **DataFrame** is a labeled table. We will give each article an ID, preserve its citation and text, and extract metadata from the citation.

Start with the title: in this workbook it is enclosed in double quotation marks. `str.extract()` applies a regex to each citation and returns the matching text. The parentheses in `"([^"]+)"` select what to keep: characters inside the quotation marks.

This is a rule for the citation format in **this file**, not a universal citation parser. We inspect missing matches instead of inventing values.

In [ ]:
news = raw.rename(columns={"Date": "citation", "Article": "article_text"}).copy()
news.index = range(1, len(news) + 1)
news.index.name = "article_id"

citations = news["citation"].astype("string").str.replace("\u00a0", " ", regex=False)
news["article_title"] = citations.str.extract(r'"([^"]+)"', expand=False)
without_title = citations.str.replace(r'"[^"]+"', "", regex=True)
display(news[["citation", "article_title"]].head(3))

,citation,article_title
article_id,,
1,"Rainsford, Sarah (6 September 2023). BBC News. ""Ukraine war: Romania reveals Russ...",Ukraine war: Romania reveals Russian drone parts hit its territory
2,"MacFarquhar, Neil (17 March 2024). The New York Times. ""Five Takeaways From Putin...",Five Takeaways From Putin's Orchestrated Win in Russia
3,"Goodman, Jack (2022-04-09). BBC News. ""Chernihiv: Are these Russia's weapons of w...",Chernihiv: Are these Russia's weapons of war?


### Extract the publication date as cited
The citations use two forms, such as `6 September 2023` and `2023-08-30`. We look for those forms **outside the quoted title** and convert them to dates that pandas can sort and plot.

We retain `date_as_cited` for checking. `errors="coerce"` leaves an unparseable or missing value as `NaT` (*Not a Time*). No date is taken from the article body: a date mentioned in a story could be an event date, update, or unrelated page material.

These are the dates stated in the supplied citations. The notebook does not independently verify original publication dates.

In [ ]:
date_pattern = r"(\d{4}-\d{2}-\d{2}|\d{1,2}\s+[A-Z][a-z]+\s+\d{4})"
news["date_as_cited"] = without_title.str.extract(date_pattern, expand=False)
news["publication_date"] = pd.to_datetime(
    news["date_as_cited"], format="mixed", errors="coerce"
)
display(news[["date_as_cited", "publication_date"]].head(5))

,date_as_cited,publication_date
article_id,,
1,6 September 2023,2023-09-06
2,17 March 2024,2024-03-17
3,2022-04-09,2022-04-09
4,3 April 2022,2022-04-03
5,25 March 2022,2022-03-25


### Extract the publishing organization as cited
A citation may list the publisher before or after the title. The supplied pattern below recognizes organization names found in this workbook; `|` means **or**. We can run this prepared rule without memorizing the publisher list.

`news_organization` means the **publisher/host named in the citation**, which can include an institutional publisher. It is not necessarily the original reporting agency. We preserve spellings such as `VOANEWS`.

In [ ]:
publisher_pattern = (
    r"(BBC News|The New York Times|Office of the High Commissioner for Human Rights|"
    r"Lawfare|Der Spiegel|The Conversation|Kyiv Post|"
    r"Center for Strategic and International Studies|The Guardian|Al Jazeera|"
    r"Radio Free Europe/Radio Liberty|The Huffington Post|Irish Times|Deutsche Welle|"
    r"Politico Europe|ukrinform\.net|Harvard Law Today|VOANEWS|Wikinews|"
    r"The Wall Street Journal|Euronews|The Kyiv Independent|The Associated Press)"
)
news["news_organization"] = without_title.str.extract(publisher_pattern, expand=False)

### Inspect the organized table and the gaps
Now each row has the four requested fields: **article title, publication date, news organization, and article text**. `citation` and `date_as_cited` remain alongside them for auditing.

`isna()` identifies missing values. Its column totals tell us how many rows need attention. Missing metadata does not imply missing article text, so we keep those rows for later reading and modeling.

In [ ]:
main_columns = ["article_title", "publication_date", "news_organization", "article_text"]
news = news[main_columns + ["citation", "date_as_cited"]]
display(news[main_columns].head(5))
print("Missing values:")
display(news[main_columns].isna().sum().to_frame("missing rows"))
needs_review = news[["article_title", "publication_date", "news_organization"]].isna().any(axis=1)
display(news.loc[needs_review, ["citation", "publication_date", "news_organization"]])

## 2. Explore the collection before modeling
### Which publishers are represented?
`value_counts()` counts records per publisher. A temporary label makes missing values visible in the chart; the stored missing metadata remains unchanged.

**Discuss:** Does this look balanced across publishers? What could a large concentration of one publisher contribute to word patterns?

In [ ]:
publisher_counts = news["news_organization"].fillna("Not stated in citation").value_counts()
ax = publisher_counts.sort_values().plot.barh(figsize=(10, 8))
ax.set_title("Articles by publishing organization as cited")
ax.set_xlabel("Number of articles")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

### Which publication years are represented?
`.dt.year` extracts the year from each parsed date. We count known years and report missing dates separately. This describes **the collected texts**, not the amount of news coverage produced in each year.

**Discuss:** Should historical background and more recent reporting be analyzed together? We will keep both in the first model and record that choice.

In [ ]:
year_counts = news["publication_date"].dropna().dt.year.value_counts().sort_index()
ax = year_counts.plot.bar(figsize=(9, 4))
ax.set_title("Publication years represented in this collection")
ax.set_xlabel("Year as cited (only represented years are shown)")
ax.set_ylabel("Number of articles")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()
print("Articles with no parsed publication date:", news["publication_date"].isna().sum())

### How long are the collected texts?
This quick word count splits on whitespace. It includes any headings or page material present in the supplied text. The model's retained-token counts will differ after preprocessing.

The histogram uses full lengths: it does not truncate or shorten the articles. `describe()` gives a compact numerical summary.

In [ ]:
news["word_count"] = news["article_text"].fillna("").str.split().str.len()
display(news["word_count"].describe().round(1).to_frame("words per article"))
ax = news["word_count"].plot.hist(bins=12, figsize=(9, 4))
ax.set_title("Lengths of the collected texts")
ax.set_xlabel("Whitespace-separated words")
ax.set_ylabel("Number of articles")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

### Exercise 1 · Read an article before representing it — 5 minutes
Change `article_id` and read the title, citation, beginning, and ending. Suggested choices: **31** (education), **41** (food shipments), or **49** (Eurovision and its war-related context). Article 31 is the starting example.

Identify two phrases suggesting what the text discusses. Does the article fit the collection you want to study? Does its text include headings, navigation, or other copied page material? Give one reason that a publisher, date, or length could matter for interpreting the model later.

The display shows excerpts; `article_text` still holds the full supplied text. Set `preview_characters = None` to print the full article after previewing its content.

In [ ]:
article_id = 31  # Change this ID and rerun this cell.
preview_characters = 700
article = news.loc[article_id]
print(article["article_title"])
print(article["citation"])
print("Words:", article["word_count"])
print("\nBEGINNING:\n", article["article_text"][:preview_characters])
print("\nENDING:\n", article["article_text"][-250:])

**Your reading notes:**  
Article ID: …  
Two phrases I noticed: …  
A scope or text-quality issue to investigate: …  
Why metadata matters here: …

## 3. Check text quality and prepare a modeling copy
We check for blank bodies and exact duplicates after whitespace normalization. We also flag a few **possible** copied-page phrases for manual review. These phrase flags are clues, not a complete detector.

The first model will exclude blank bodies and repeated exact bodies after their first occurrence. It retains short records, missing metadata, historical background, and flagged page material so those decisions are visible. We make no claim that the first pass is a fully cleaned research corpus.

In [ ]:
text_key = news["article_text"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
news["empty_text"] = text_key.eq("")
news["duplicate_text"] = text_key.duplicated() & ~news["empty_text"]
news["short_text"] = news["word_count"] < 200
news["possible_site_text"] = news["article_text"].str.contains(
    r"all rights reserved|register now|disclosure statement|up pointing triangle",
    case=False, regex=True, na=False
)
flags = ["empty_text", "duplicate_text", "short_text", "possible_site_text"]
display(news[flags].sum().to_frame("flagged records"))
display(news.loc[news[flags].any(axis=1), ["article_title", "word_count"] + flags])

### Choose the modeling population explicitly
We use all nonempty, nonduplicate texts in the workbook. `model_text` is a separate, whitespace-normalized copy of the article body. We do not add the citation, extracted title, publisher, date, or our own topic names to it.

Some supplied bodies already contain headings and bylines; those remain until manually reviewed. Keeping `article_text` unchanged lets us compare any future cleaning decision with the source.

The article IDs remain the same even if rows are excluded. ID 31 refers to the same source row throughout the notebook.

In [ ]:
keep = ~news["empty_text"] & ~news["duplicate_text"]
model_news = news.loc[keep].copy()
model_news["model_text"] = text_key.loc[model_news.index]
news["included_in_model"] = news.index.isin(model_news.index)
print("Articles in the workbook:", len(news))
print("Articles in this model:", len(model_news))
print("Excluded article IDs:", news.index[~news["included_in_model"]].tolist())

### Turn article bodies into Bag-of-Words counts
Our preprocessing policy is explicit: lowercase; use alphabetic word tokens of at least two letters; remove the library's English stopwords plus **said, says, say**; keep words in at least two documents and in no more than 90% of documents; retain at most 3,000 terms by corpus frequency.

These are starting choices, not rules that every project must adopt. We do not use lemmas, bigrams, or TF-IDF in this LDA demo. The vocabulary is learned from **article bodies only**. `min_df=2` counts documents, not a word's total repetitions.

A word that is removed may still matter to the research question. The next cells make the selected features visible before fitting the model.

In [ ]:
stop_words = sorted(ENGLISH_STOP_WORDS.union({"said", "says", "say"}))
vectorizer = CountVectorizer(
    stop_words=stop_words, min_df=2, max_df=0.9, max_features=3000,
    token_pattern=r"\b[a-zA-Z]{2,}\b"
)
counts = vectorizer.fit_transform(model_news["model_text"])
terms = vectorizer.get_feature_names_out()
print("Count matrix:", counts.shape, "= articles × vocabulary terms")
if (counts.getnnz(axis=1) == 0).any():
    raise ValueError("A text has no retained features. Review preprocessing before fitting LDA.")

### Read a small window of the feature table
For this small classroom corpus, we can display counts as a DataFrame. The selected words are **only a display window**; they do not define the model's topics or restrict its full vocabulary.

Then inspect the most frequent retained terms. Common words help us understand the input, but a frequency list by itself is not a topic model. Large real datasets should remain in sparse form instead of using `toarray()` on the entire matrix.

In [ ]:
count_table = pd.DataFrame(counts.toarray(), index=model_news.index, columns=terms)
view_words = ["children", "school", "grain", "food", "energy", "nuclear", "law", "international"]
display(count_table.reindex(index=[31, 41, 52, 66], columns=view_words))
term_totals = count_table.sum(axis=0).sort_values(ascending=False)
display(term_totals.head(12).to_frame("total occurrences"))

## 4. Fit LDA: counts in, two tables out
We choose **four topics** for an initial classroom model. Four is a setting to investigate; it is not a discovered number of themes in the war or its coverage.

`fit_transform(counts)` learns patterns from the word counts and returns one topic mixture per article. We supply **no topic labels**. `random_state=42` makes the initial randomness repeatable; `max_iter=50` allows up to 50 passes of the chosen batch algorithm. Reproducibility is not a test of topic quality.

The two outputs mirror the lecture: **words characterizing each topic**, and **topic mixtures within articles**. We do not calculate classification accuracy because no validated target topic labels are supplied.

In [ ]:
number_of_topics = 4
lda = LatentDirichletAllocation(
    n_components=number_of_topics, learning_method="batch", max_iter=50, random_state=42
)
mixtures = lda.fit_transform(counts)
topic_names = []
for number in range(1, number_of_topics + 1):
    topic_names.append(f"Topic {number}")
document_topics = pd.DataFrame(mixtures, index=model_news.index, columns=topic_names)
print("Document–topic table:", document_topics.shape)

### Output 1: Which words characterize each topic?
`lda.components_` contains topic–word weights. Dividing each row by its row total turns those weights into probabilities over the full selected vocabulary. Each complete row sums to 1.

We display the ten highest-probability words for each topic. **A word can have probability in several topics.** The ten displayed words do not themselves sum to 1. Topic numbers are identifiers, not names or an importance ranking.

In [ ]:
topic_words = pd.DataFrame(lda.components_, index=topic_names, columns=terms)
topic_words = topic_words.div(topic_words.sum(axis=1), axis="index")
rows = []
for topic in topic_names:
    top_words = topic_words.loc[topic].nlargest(10).index
    rows.append({"topic": topic, "top words": ", ".join(top_words)})
topic_summary = pd.DataFrame(rows).set_index("topic")
with pd.option_context("display.max_colwidth", None):
    display(topic_summary)

### Output 2: How do topics mix within each article?
Each row contains an article's estimated topic proportions. We multiply by 100 **only for display**, making them easier to read as percentages. We keep the unrounded values for subsequent calculations.

Read these as estimated mixtures under this model and preprocessing policy. They are not percentages of sentences, factual-accuracy scores, or confidence that a human label is correct. Some articles may have a very concentrated mixture; the short lecture example does not guarantee equally clear mixtures in real texts.

In [ ]:
mixture_preview = model_news[["article_title", "news_organization"]].join(100 * document_topics)
print("Estimated topic percentages; each complete row totals 100% before rounding.")
display(mixture_preview.loc[[31, 41, 52, 66]].round(1))
print("First three row totals:", document_topics.sum(axis=1).head(3).round(6).tolist())

### Zoom in on one article's mixture
Article **66** is about a front-line school system and appears as a mixed document in the saved demonstration. The chart uses the fitted values from this notebook, not the invented percentages from the lecture.

Change `mixture_article_id` to inspect another modeled text. Use the title to orient yourself, then return to the supplied article before deciding what the mixture means. Its title was not added as a separate model feature.

In [ ]:
mixture_article_id = 66
print(model_news.loc[mixture_article_id, "article_title"])
shares = 100 * document_topics.loc[mixture_article_id]
ax = shares.plot.barh(figsize=(8, 4))
ax.set_title(f"Article {mixture_article_id}: estimated topic mixture")
ax.set_xlim(0, 100)
ax.set_xlabel("Estimated topic proportion (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 5. Read documents to interpret topics
### Exercise 2 · Test a tentative topic name — 10 minutes
Select a topic from the top-word table. Before naming it, inspect the articles with the largest estimated proportions of that topic. The table below ranks texts by **modeled topic proportion**, not factual reliability or writing quality.

Read at least two of those articles. Write a tentative topic name, cite two supporting words or passages, and identify something the name leaves out. A mixed or unhelpful topic is a result to investigate, not something we should disguise with a confident label.

For a starting point, inspect `Topic 3`. Preview article content before displaying long passages.

In [ ]:
topic_to_inspect = "Topic 3"  # Choose a topic from the displayed table.
print("Top words:", topic_summary.loc[topic_to_inspect, "top words"])
top_ids = document_topics[topic_to_inspect].nlargest(4).index
top_articles = model_news.loc[top_ids, ["article_title", "publication_date", "news_organization"]].copy()
top_articles["topic_percent"] = 100 * document_topics.loc[top_ids, topic_to_inspect]
display(top_articles.round(1))

In [ ]:
reading_id = 57  # Then choose a second ID from the topic's article table above.
reading = model_news.loc[reading_id]
print(reading["article_title"])
print(reading["citation"])
print(reading["article_text"][:900])
print("\nTopic percentages:")
display((100 * document_topics.loc[reading_id]).round(1).to_frame("percent"))

**Your interpretation notes:**  
Topic number and tentative name: …  
Two article IDs read: …  
Evidence supporting the name: …  
An article or passage the name does not explain well: …  
One change or additional check to try: …

## 6. Save the tables and explain the limits
The exports preserve the article IDs so we can reconnect every model row to its citation. The structured dataset keeps **all uploaded rows**, flags, and unchanged source text. The document–topic export contains only modeled rows. Topic–word probabilities cover the full selected vocabulary.

Save copies in your own course/project space and retain the source attributions; the workbook does not establish permission to redistribute its full texts publicly.

In [ ]:
news.to_csv("Ukraine_News_Structured.csv", encoding="utf-8-sig", date_format="%Y-%m-%d")
model_news[["article_title", "publication_date", "news_organization"]].join(document_topics).to_csv(
    "Ukraine_News_Document_Topics.csv", encoding="utf-8-sig", date_format="%Y-%m-%d"
)
topic_words.to_csv("Ukraine_News_Topic_Words.csv", encoding="utf-8-sig", index_label="topic")
print("Saved three CSV files. Use the Colab Files sidebar to download them.")

### Exit questions
How is a word-frequency table different from a topic model? What does one row of the document–topic table describe? Which choice about sources, dates, scope, or cleaning would you revisit before making a research claim?

This is an exploratory analysis of a small, purposively collected set of texts. It cannot establish the most important events, the dominant themes in all news coverage, or whether an article is factually correct. We have no validated topic labels and have not established an optimal number of topics.

The lecture's central idea carries through: **read the documents, inspect the word distributions, and keep document mixtures available instead of replacing every article with one label.**

---
## Optional after the main demo · Change only the number of topics
Try three topics while holding the documents, vocabulary, and random seed fixed. We also copy the baseline's prior settings, which would otherwise change with the library defaults when K changes. This does not identify the “correct” topic count.

Read the new top-word table and supporting articles. Topic numbers are local to a fitted model: **Topic 1 in a three-topic model is not automatically Topic 1 in a four-topic model.** We use new variables so the baseline tables above remain unchanged.

In [ ]:
alternative_k = 3
alternative_lda = LatentDirichletAllocation(
    n_components=alternative_k, learning_method="batch", max_iter=50, random_state=42,
    doc_topic_prior=lda.doc_topic_prior_, topic_word_prior=lda.topic_word_prior_
)
alternative_mixtures = alternative_lda.fit_transform(counts)
alternative_rows = []
for number, weights in enumerate(alternative_lda.components_, start=1):
    words = pd.Series(weights, index=terms).nlargest(10).index
    alternative_rows.append({"topic in this model": number, "top words": ", ".join(words)})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(alternative_rows))

### Optional: inspect source composition after modeling
Compare **document counts**, not source quality. A large concentration of one publisher in a topic may reflect collection design or writing style, as well as subject matter.

The following table assigns each article to its largest-proportion topic only to create a compact count table. It discards mixture information, so keep the document–topic table for interpretation. Raw counts should not be compared as if publisher sample sizes were equal.

In [ ]:
source_topics = pd.DataFrame({
    "publisher as cited": model_news["news_organization"].fillna("Not stated in citation"),
    "largest-proportion topic": document_topics.idxmax(axis=1)
})
display(pd.crosstab(source_topics["publisher as cited"], source_topics["largest-proportion topic"]))

### Sources and methods
**Data source:** `Ukraine_News_Dataset.xlsx`, worksheet `Articles`.

**Course basis:** the Week 6 topic-modeling lecture's Bag-of-Words input, topic–word distributions, document mixtures, and document-reading checks.

**Technical documentation**:

- [pandas: read Excel files](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html)
- [pandas: extract regex groups](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html)
- [pandas: convert mixed date formats](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [scikit-learn: CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [scikit-learn: LatentDirichletAllocation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)
- [scikit-learn: topic extraction using counts for LDA](https://scikit-learn.org/stable/auto_examples/applications/plot_topics_extraction_with_nmf_lda.html)